# PyTorch Tabular Temperature Forecasting

This notebook adds a PyTorch tabular supervised-learning workflow on top of the canonical Open-Meteo hourly table. It stays CPU-friendly while comparing several small MLP regressors side by side.

The workflow is:

- reuse the canonical hourly table and temperature-forecasting feature recipe
- build a leakage-safe chronological split
- compare compact MLP baselines for next-hour `temperature_2m`
- inspect learning curves and a short prediction window

In [ ]:
from __future__ import annotations

import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from open_meteo import DEFAULT_DATA_DIR, build_hourly_table

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
torch.set_num_threads(min(4, os.cpu_count() or 1))
device = torch.device('cpu')
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print({'device': str(device), 'torch_version': torch.__version__})

## Load the canonical hourly table

The notebook uses the same aligned hourly table and near-term temperature target as the sklearn forecasting notebook so the comparisons stay directly compatible.

In [ ]:
table = build_hourly_table(DEFAULT_DATA_DIR)
column_arrays = {name: np.asarray(values, dtype=float) for name, values in table.columns.items()}
timestamps = np.asarray(table.timestamps)

required_columns = [
    'misc__temperature_2m',
    'misc__relative_humidity_2m',
    'misc__dew_point_2m',
    'misc__apparent_temperature',
    'misc__pressure_msl',
    'misc__precipitation',
    'misc__vapour_pressure_deficit',
    'wind__wind_speed_10m',
    'wind__wind_gusts_10m',
    'cloud_cover__cloud_cover',
    'solar_radiation_vars__shortwave_radiation',
    'additional_vars_and_options__is_day',
    'european_air_quality_index__european_aqi',
]
missing = [name for name in required_columns if name not in column_arrays]
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

def build_calendar_features(time_values: np.ndarray) -> dict[str, np.ndarray]:
    hours = np.asarray([int(value[11:13]) for value in time_values], dtype=float)
    months = np.asarray([int(value[5:7]) for value in time_values], dtype=float)
    days = np.asarray([int(value[8:10]) for value in time_values], dtype=float)
    day_of_year = np.asarray([
        int((np.datetime64(value[:10]) - np.datetime64(value[:4] + '-01-01')) / np.timedelta64(1, 'D')) + 1
        for value in time_values
    ], dtype=float)
    return {
        'hour': hours,
        'month': months,
        'day': days,
        'day_of_year': day_of_year,
        'hour_sin': np.sin(2.0 * np.pi * hours / 24.0),
        'hour_cos': np.cos(2.0 * np.pi * hours / 24.0),
        'day_of_year_sin': np.sin(2.0 * np.pi * day_of_year / 366.0),
        'day_of_year_cos': np.cos(2.0 * np.pi * day_of_year / 366.0),
    }

calendar_features = build_calendar_features(timestamps)
print(f'Rows: {len(timestamps):,}')
print(f'Columns: {len(column_arrays):,}')
print(f'Start: {timestamps[0]}')
print(f'End:   {timestamps[-1]}')

## Build a tabular supervised dataset

We predict the next hour of `temperature_2m` from the current weather state, calendar features, and recent temperature lags. The benchmark uses a recent subset so the notebook stays practical on CPU.

In [ ]:
target_name = 'misc__temperature_2m'
base_feature_names = [
    'misc__temperature_2m',
    'misc__relative_humidity_2m',
    'misc__dew_point_2m',
    'misc__apparent_temperature',
    'misc__pressure_msl',
    'misc__precipitation',
    'misc__vapour_pressure_deficit',
    'wind__wind_speed_10m',
    'wind__wind_gusts_10m',
    'cloud_cover__cloud_cover',
    'solar_radiation_vars__shortwave_radiation',
    'additional_vars_and_options__is_day',
    'european_air_quality_index__european_aqi',
]
lag_hours = [1, 3, 6, 24]
forecast_horizon = 1
max_lag = max(lag_hours)
row_start = max_lag
row_end = len(timestamps) - forecast_horizon
max_rows = 30000

feature_blocks = []
feature_names = []

for name in base_feature_names:
    feature_blocks.append(column_arrays[name][row_start:row_end])
    feature_names.append(name)

for name, values in calendar_features.items():
    feature_blocks.append(values[row_start:row_end])
    feature_names.append(name)

target_values = column_arrays[target_name]
for lag in lag_hours:
    feature_blocks.append(target_values[row_start - lag:row_end - lag])
    feature_names.append(f'{target_name}__lag_{lag}h')

feature_matrix = np.column_stack(feature_blocks).astype(np.float32)
target = target_values[row_start + forecast_horizon:row_end + forecast_horizon].astype(np.float32)
sample_timestamps = timestamps[row_start:row_end]

valid_mask = np.isfinite(feature_matrix).all(axis=1) & np.isfinite(target)
X = feature_matrix[valid_mask]
y = target[valid_mask]
time_index = sample_timestamps[valid_mask]

if len(y) > max_rows:
    X = X[-max_rows:]
    y = y[-max_rows:]
    time_index = time_index[-max_rows:]

print(f'Supervised rows: {len(y):,}')
print(f'Feature count: {X.shape[1]}')
print('First features:', feature_names[:8])
print(f'Subset start: {time_index[0]}')
print(f'Subset end:   {time_index[-1]}')

In [ ]:
def chronological_split(X: np.ndarray, y: np.ndarray, time_index: np.ndarray, train_ratio: float = 0.7, valid_ratio: float = 0.15):
    total = len(y)
    train_end = int(total * train_ratio)
    valid_end = int(total * (train_ratio + valid_ratio))
    return {
        'X_train': X[:train_end],
        'y_train': y[:train_end],
        't_train': time_index[:train_end],
        'X_valid': X[train_end:valid_end],
        'y_valid': y[train_end:valid_end],
        't_valid': time_index[train_end:valid_end],
        'X_test': X[valid_end:],
        'y_test': y[valid_end:],
        't_test': time_index[valid_end:],
    }

split = chronological_split(X, y, time_index)
x_mean = split['X_train'].mean(axis=0)
x_std = split['X_train'].std(axis=0) + 1e-6
y_mean = split['y_train'].mean()
y_std = split['y_train'].std() + 1e-6

def normalize_features(values: np.ndarray) -> np.ndarray:
    return ((values - x_mean) / x_std).astype(np.float32)

def normalize_targets(values: np.ndarray) -> np.ndarray:
    return ((values - y_mean) / y_std).astype(np.float32)

def denormalize_targets(values: np.ndarray) -> np.ndarray:
    return values * y_std + y_mean

def to_tensor_dataset(X_values: np.ndarray, y_values: np.ndarray) -> TensorDataset:
    return TensorDataset(
        torch.from_numpy(normalize_features(X_values)),
        torch.from_numpy(normalize_targets(y_values)),
    )

train_dataset = to_tensor_dataset(split['X_train'], split['y_train'])
valid_dataset = to_tensor_dataset(split['X_valid'], split['y_valid'])
test_dataset = to_tensor_dataset(split['X_test'], split['y_test'])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=512, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

for name in ['train', 'valid', 'test']:
    print(name, split[f'X_{name}'].shape, split[f't_{name}'][0], split[f't_{name}'][-1])

## Define compact PyTorch MLP regressors

These models are intentionally small so the benchmark stays CPU-friendly while still comparing shallow and deeper tabular architectures.

In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, input_size: int, hidden_layers: list[int], dropout: float = 0.0):
        super().__init__()
        layers = []
        prev_size = input_size
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x).squeeze(-1)

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(mean_squared_error(y_true, y_pred, squared=False)),
        'r2': float(r2_score(y_true, y_pred)),
    }

def evaluate_model(model: nn.Module, loader: DataLoader) -> tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    loss_fn = nn.MSELoss()
    losses = []
    preds = []
    targets = []
    with torch.no_grad():
        for features, target in loader:
            features = features.to(device)
            target = target.to(device)
            output = model(features)
            loss = loss_fn(output, target)
            losses.append(loss.item())
            preds.append(output.cpu().numpy())
            targets.append(target.cpu().numpy())
    pred_array = denormalize_targets(np.concatenate(preds))
    target_array = denormalize_targets(np.concatenate(targets))
    return float(np.mean(losses)), pred_array, target_array

def train_model(model: nn.Module, train_loader: DataLoader, valid_loader: DataLoader, epochs: int = 12, lr: float = 1e-3, weight_decay: float = 1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    history = []
    best_valid_loss = float('inf')
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for features, target in train_loader:
            features = features.to(device)
            target = target.to(device)
            optimizer.zero_grad()
            output = model(features)
            loss = loss_fn(output, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = float(np.mean(batch_losses))
        valid_loss, _, _ = evaluate_model(model, valid_loader)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'valid_loss': valid_loss})

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return history

## Compare PyTorch MLP baselines

The benchmark compares multiple small tabular networks side by side so you can see whether extra depth helps on this temperature-forecasting setup.

In [ ]:
benchmark_specs = {
    'mlp_shallow': {'hidden_layers': [64], 'dropout': 0.0},
    'mlp_medium': {'hidden_layers': [128, 64], 'dropout': 0.05},
    'mlp_deep': {'hidden_layers': [128, 96, 48], 'dropout': 0.1},
}

benchmark_rows = []
histories = {}
test_predictions = {}

for model_name, spec in benchmark_specs.items():
    model = MLPRegressor(
        input_size=X.shape[1],
        hidden_layers=spec['hidden_layers'],
        dropout=spec['dropout'],
    ).to(device)
    history = train_model(model, train_loader, valid_loader, epochs=12, lr=1e-3, weight_decay=1e-4)
    valid_loss, valid_pred, valid_true = evaluate_model(model, valid_loader)
    test_loss, test_pred, test_true = evaluate_model(model, test_loader)
    valid_scores = regression_metrics(valid_true, valid_pred)
    test_scores = regression_metrics(test_true, test_pred)
    benchmark_rows.append({
        'model': model_name,
        'valid_mae': valid_scores['mae'],
        'valid_rmse': valid_scores['rmse'],
        'test_mae': test_scores['mae'],
        'test_rmse': test_scores['rmse'],
        'test_r2': test_scores['r2'],
    })
    histories[model_name] = history
    test_predictions[model_name] = test_pred

for row in sorted(benchmark_rows, key=lambda item: item['test_rmse']):
    print(row)

In [ ]:
best_model_name = min(benchmark_rows, key=lambda item: item['test_rmse'])['model']
plot_count = min(24 * 7, len(split['y_test']))

fig, ax = plt.subplots()
ax.plot(split['t_test'][:plot_count], split['y_test'][:plot_count], label='actual', linewidth=2)
ax.plot(split['t_test'][:plot_count], test_predictions[best_model_name][:plot_count], label=best_model_name, linewidth=1.8)
ax.set_title('Next-hour temperature forecast on the first week of the test split')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Temperature (C)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()

fig, axes = plt.subplots(1, len(histories), figsize=(5 * len(histories), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (model_name, history) in zip(axes, histories.items()):
    epochs = [row['epoch'] for row in history]
    train_loss = [row['train_loss'] for row in history]
    valid_loss = [row['valid_loss'] for row in history]
    ax.plot(epochs, train_loss, label='train')
    ax.plot(epochs, valid_loss, label='valid')
    ax.set_title(model_name)
    ax.set_xlabel('Epoch')
    ax.legend()
axes[0].set_ylabel('MSE loss on normalized target')
plt.tight_layout()

## Suggested follow-up experiments

- add rolling-window summaries and finite-difference features
- compare these MLP baselines against the sklearn temperature regressors
- switch the same tabular pipeline to `european_aqi` or `wind_speed_10m`
- try a wider hidden layer only if the CPU runtime still looks comfortable